## Environment Setup

This notebook evaluates a Large Language Model (LLM) for sentiment classification of Amazon product reviews.

The experiment uses:

- **Ollama** to run a local LLM (Llama 3.1)
- **Scikit-learn** to compute evaluation metrics
- **Pandas** for dataset handling

Running the model locally avoids API rate limits and allows the experiment to be reproducible.

In [57]:
!pip install dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [36]:
import sys
!{sys.executable} -m pip install -U ollama scikit-learn

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.0 MB 8.3 MB/s eta 0:00:01
   ------------------------------ --------- 6.0/8.0 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 11.7 MB/s  0:00:00

  Attempting uninstall: scikit-learn

    Found existing installation: scikit-learn 1.7.2

   ---------------------------------------- 0/2 [scikit-learn]
    Uninstalling scikit-learn-1.7.2:
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
      Successfully uninstalled scikit-learn-1.7.2
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ------------

  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Required Libraries

We import the libraries required for:

- Data processing (`pandas`)
- Timing and execution control (`time`)
- Text normalization (`re`)
- Model evaluation (`scikit-learn`)
- 
-

In [61]:
import re
import time
import pandas as pd

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import ollama

## Load Dataset

The dataset used in this experiment is a cleaned subset of Amazon product reviews.

Each review contains:

- `reviewText` → the text of the review
- `sentiment` → the ground truth label derived from the star rating

Sentiment labels were created as:

- **Positive** → ratings 4–5  
- **Negative** → ratings 1–2  
- Neutral reviews were removed to create a binary classification task.

In [38]:
DATA_PATH = "../data/amazon_reviews_llm_ready.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df["sentiment"].value_counts())
df.head()

(648, 5)
sentiment
Positive    324
Negative    324
Name: count, dtype: int64


,reviewText,overall,sentiment,word_len,char_len
0,Installed as sublimental memory for Samsung Ta...,5.0,Positive,84,418
1,NOT GOOD REPEATEDLY SELF EJECTS FROM NEW GALAX...,1.0,Negative,33,193
2,Installed this card in my wife's cel phone. I...,5.0,Positive,22,113
3,I picked up this storage device to use in my p...,5.0,Positive,44,202
4,After I inserted the micro SD card to the adap...,2.0,Negative,111,630


## Sampling Reviews for Evaluation

Running a local LLM on the entire dataset would take a long time.

Instead, we sample a **subset of reviews** to evaluate the model.

For this experiment we make a test case of **100 reviews** to estimate model performance, but we end up taking half due to token costs.

In [39]:
N_SAMPLES = 100

df_eval = df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)
print(df_eval["sentiment"].value_counts())

sentiment
Negative    54
Positive    46
Name: count, dtype: int64


## Prompt Engineering

Large Language Models can perform new tasks using **prompt engineering**.

In this experiment we use **few-shot prompting**, where a few labeled examples are included in the prompt to guide the model.

The model is instructed to output only one of two labels:

- Positive
- Negative

In [40]:
ZERO_SHOT_PROMPT = """Classify the sentiment of this Amazon product review.

Return exactly one word:
Positive
or
Negative

Review:
{review}
"""

FEW_SHOT_PROMPT = """Classify the sentiment of this Amazon product review.

Return exactly one word:
Positive
or
Negative

Examples:
Review: "Works perfectly, great quality and fast delivery."
Sentiment: Positive

Review: "Stopped working after a week. Very disappointed."
Sentiment: Negative

Review: "Excellent value for the price."
Sentiment: Positive

Now classify this review:
Review: {review}
Sentiment:
"""

## Output Normalization

LLMs sometimes return additional text instead of a single label.

To ensure consistent evaluation, we normalize the model output so that responses map to either:

- **Positive**
- **Negative**
- **Unknown** (if the output cannot be interpreted)

In [41]:
def normalize_prediction(text: str) -> str:
    if text is None:
        return "Unknown"

    t = str(text).strip().lower()

    if "positive" in t:
        return "Positive"
    if "negative" in t:
        return "Negative"

    # fallback: first token check
    first = re.split(r"[\s\.\!\?,:\n\r]+", t)[0]
    if first == "positive":
        return "Positive"
    if first == "negative":
        return "Negative"

    return "Unknown"

## Local LLM Inference with Ollama

The experiment uses **Llama 3.1** running locally through **Ollama**.

Each review is sent to the model with the prompt, and the model predicts the sentiment.

Using a local model avoids API rate limits and allows us to test the experiment offline.

In [44]:
OLLAMA_MODEL = "llama3.1"

def predict_ollama(review: str, prompt_template: str, model: str = OLLAMA_MODEL) -> str:
    prompt = prompt_template.format(review=review)

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={
            "temperature": 0
        }
    )

    return response["message"]["content"]

## Sentiment Classification Experiment

The model is now evaluated on **50 Amazon reviews**.

For each review:

1. The text is sent to the LLM
2. The model predicts the sentiment
3. The prediction is normalized
4. The result is stored

Progress updates are printed during execution because local LLM inference can take several minutes.

In [45]:
sample_review = df_eval.loc[0, "reviewText"]

raw = predict_ollama(sample_review, FEW_SHOT_PROMPT)
pred = normalize_prediction(raw)

print("Review:", sample_review)
print("Raw response:", raw)
print("Normalized prediction:", pred)
print("Ground truth:", df_eval.loc[0, "sentiment"])

Review: This card does not work on droid razr phones. Whenever I try to transfer files on the card, it makes the phone freeze completely. I have tried every possibility to get this to work, but this type of card is worthless because it does not work due to its high capacity on phones. Every workaround I have tried to get this card to work, it still causes my phone to freeze.
Raw response: Negative
Normalized prediction: Negative
Ground truth: Negative


## Sentiment Classification Experiment

The model is now evaluated on **50 Amazon reviews**.

For each review:

1. The text is sent to the LLM
2. The model predicts the sentiment
3. The prediction is normalized
4. The result is stored

Progress updates are printed during execution because local LLM inference can take several minutes.

In [49]:
import time

# sample 50 reviews
N_SAMPLES = 50
df_eval = df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)

# initialize empty columns
df_eval["pred_raw"] = ""
df_eval["pred"] = ""

start_time = time.time()

for i, review in enumerate(df_eval["reviewText"]):
    try:
        raw = predict_ollama(review, FEW_SHOT_PROMPT)
        pred = normalize_prediction(raw)
    except Exception as e:
        raw = f"ERROR: {e}"
        pred = "Unknown"

    # save directly into the dataframe row
    df_eval.loc[i, "pred_raw"] = raw
    df_eval.loc[i, "pred"] = pred

    # progress update every 5 reviews
    if (i + 1) % 5 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (N_SAMPLES - (i + 1))

        print(
            f"{i+1}/{N_SAMPLES} done | "
            f"elapsed: {elapsed/60:.1f} min | "
            f"ETA: {remaining/60:.1f} min"
        )

        # checkpoint save
        df_eval.to_csv("../results/ollama_partial_results.csv", index=False)

# final save
df_eval.to_csv("../results/ollama_results_final.csv", index=False)
print("\nFinished evaluation.")

5/50 done | elapsed: 0.7 min | ETA: 6.6 min
10/50 done | elapsed: 1.7 min | ETA: 6.7 min
15/50 done | elapsed: 2.8 min | ETA: 6.5 min
20/50 done | elapsed: 3.8 min | ETA: 5.7 min
25/50 done | elapsed: 4.5 min | ETA: 4.5 min
30/50 done | elapsed: 5.7 min | ETA: 3.8 min
35/50 done | elapsed: 6.7 min | ETA: 2.9 min
40/50 done | elapsed: 8.1 min | ETA: 2.0 min
45/50 done | elapsed: 9.5 min | ETA: 1.1 min
50/50 done | elapsed: 10.2 min | ETA: 0.0 min

Finished evaluation.


## Model Evaluation

After predictions are generated, we compare them with the ground truth sentiment labels.

The following metrics are calculated:

- **Accuracy** – overall classification performance
- **Confusion Matrix** – breakdown of correct and incorrect predictions
- **Classification Report** – precision, recall, and F1-score

In [50]:
valid = df_eval[df_eval["pred"] != "Unknown"].copy()

y_true = valid["sentiment"]
y_pred = valid["pred"]

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=["Positive", "Negative"]))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=["Positive", "Negative"]))

Accuracy: 0.92

Confusion Matrix:
[[24  4]
 [ 0 22]]

Classification Report:
              precision    recall  f1-score   support

    Positive       1.00      0.86      0.92        28
    Negative       0.85      1.00      0.92        22

    accuracy                           0.92        50
   macro avg       0.92      0.93      0.92        50
weighted avg       0.93      0.92      0.92        50



## Misclassified Examples

To better understand model limitations, we examine several reviews that were classified incorrectly.

This helps identify cases where sentiment may be ambiguous or where the model misunderstood the review context.

In [51]:
errors = valid[valid["pred"] != valid["sentiment"]][
    ["reviewText", "sentiment", "pred", "pred_raw"]
].head(10)

errors

,reviewText,sentiment,pred,pred_raw
1,came in fast and managed to get it at a gold b...,Positive,Negative,Negative \n\nThe reviewer mentions that they g...
17,Pros: 32 GBs more for my tablet. Works great.C...,Positive,Negative,Negative
32,SANDISK is now processing my RMA. It was used...,Positive,Negative,Negative
41,I tested it in several UHS-1 compliant devices...,Positive,Negative,Negative


## New Prompt With More Shots

To test against our first results, in order to improve accuracy, we will optimize our prompt and see if it can improve our accuracy.

In [53]:
FEW_SHOT_PROMPT_V2 = """You are an expert sentiment classifier for Amazon product reviews.

Task:
Classify each review as either Positive or Negative.

Guidelines:
- Positive = the customer is satisfied, recommends the product, or describes good performance.
- Negative = the customer is dissatisfied, reports failure, poor quality, or bad performance.
- Output exactly one word only: Positive or Negative
- Do not explain.
- Do not output anything else.

Examples:

Review: "Works perfectly, great quality and fast delivery."
Sentiment: Positive

Review: "Stopped working after a week. Very disappointed."
Sentiment: Negative

Review: "Excellent value for the price. I would buy it again."
Sentiment: Positive

Review: "The card corrupted my files and made my phone freeze."
Sentiment: Negative

Now classify this review:

Review: {review}
Sentiment:"""

In [54]:
import time

# sample 50 reviews
N_SAMPLES = 50
df_eval = df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)

# initialize empty columns
df_eval["pred_raw"] = ""
df_eval["pred"] = ""

start_time = time.time()

for i, review in enumerate(df_eval["reviewText"]):
    try:
        raw = predict_ollama(review, FEW_SHOT_PROMPT_V2)
        pred = normalize_prediction(raw)
    except Exception as e:
        raw = f"ERROR: {e}"
        pred = "Unknown"

    # save directly into the dataframe row
    df_eval.loc[i, "pred_raw"] = raw
    df_eval.loc[i, "pred"] = pred

    # progress update every 5 reviews
    if (i + 1) % 5 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (N_SAMPLES - (i + 1))

        print(
            f"{i+1}/{N_SAMPLES} done | "
            f"elapsed: {elapsed/60:.1f} min | "
            f"ETA: {remaining/60:.1f} min"
        )

        # checkpoint save
        df_eval.to_csv("../results/ollama_partial_results_v2.csv", index=False)

# final save
df_eval.to_csv("../results/ollama_results_final_v2.csv", index=False)
print("\nFinished evaluation.")

5/50 done | elapsed: 1.1 min | ETA: 10.2 min
10/50 done | elapsed: 2.4 min | ETA: 9.6 min
15/50 done | elapsed: 3.4 min | ETA: 7.9 min
20/50 done | elapsed: 4.5 min | ETA: 6.7 min
25/50 done | elapsed: 5.4 min | ETA: 5.4 min
30/50 done | elapsed: 6.8 min | ETA: 4.6 min
35/50 done | elapsed: 8.1 min | ETA: 3.5 min
40/50 done | elapsed: 9.9 min | ETA: 2.5 min
45/50 done | elapsed: 11.7 min | ETA: 1.3 min
50/50 done | elapsed: 12.6 min | ETA: 0.0 min

Finished evaluation.


In [55]:
# Keep only valid predictions
valid = df_eval[df_eval["pred"].isin(["Positive", "Negative"])].copy()

print("Valid predictions:", len(valid), "out of", len(df_eval))
print("\nPrediction counts:")
print(valid["pred"].value_counts())

y_true = valid["sentiment"]
y_pred = valid["pred"]

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=["Positive", "Negative"])

print("\nAccuracy:", round(acc, 4))
print("\nConfusion Matrix (rows=true, cols=pred):")
print(cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=["Positive", "Negative"]))

Valid predictions: 50 out of 50

Prediction counts:
pred
Positive    27
Negative    23
Name: count, dtype: int64

Accuracy: 0.98

Confusion Matrix (rows=true, cols=pred):
[[27  1]
 [ 0 22]]

Classification Report:
              precision    recall  f1-score   support

    Positive       1.00      0.96      0.98        28
    Negative       0.96      1.00      0.98        22

    accuracy                           0.98        50
   macro avg       0.98      0.98      0.98        50
weighted avg       0.98      0.98      0.98        50



In [56]:
cm_df = pd.DataFrame(
    cm,
    index=["Actual Positive", "Actual Negative"],
    columns=["Predicted Positive", "Predicted Negative"]
)

cm_df

,Predicted Positive,Predicted Negative
Actual Positive,27,1
Actual Negative,0,22
